In [0]:
df = spark.table("lesson_db.orders")

display(df.show(1))


displayHTML(df.printSchema())

+--------+-----------+----------+-------+------+----+----------+
|order_id|customer_id|product_id|country|amount|year|order_date|
+--------+-----------+----------+-------+------+----+----------+
|       0|         41|        23|     FR|515.62|2023|2024-03-21|
+--------+-----------+----------+-------+------+----+----------+
only showing top 1 row
root
 |-- order_id: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- order_date: date (nullable = true)



In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
w = Window.partitionBy("customer_id").orderBy(F.desc("order_date"))

df = spark.table("lesson_db.orders")

# df.count()

display(df.withColumn("rn", F.row_number().over(w)).take(1))

order_id,customer_id,product_id,country,amount,year,order_date,rn
354828,18,74,JP,677.63,2023,2025-12-28,1


In [0]:

#create a set of records with [1, 2, 2, 2].

records = [(1,), (2,), (2,), (2,)]
records = spark.createDataFrame(records, ["value"])

seen, dupes = set(), 0
for r in records.collect():
    v = r['value']
    if v in seen:
        dupes += 1
    seen.add(v)

# Trace it on [1, 2, 2, 2]. How many dupes does it report?

dupes

2

In [0]:
import re
from abc import ABC, abstractmethod

# 1. The Base Rule Interface
class DQRule(ABC):
    @abstractmethod
    def evaluate(self, records: list) -> dict:
        """Each rule must evaluate the data and return its own metrics."""
        pass

# 2. Concrete Implementation of the Regex Check
class RegexRule(DQRule):
    def __init__(self, column: str, pattern: str):
        self.column = column
        self.pattern = re.compile(pattern)

    def evaluate(self, records: list) -> dict:
        failures = 0
        for r in records:
            v = str(r.get(self.column, ""))
            if not self.pattern.match(v):
                failures += 1
        return {"rule": "RegexCheck", "column": self.column, "failures": failures}

# 3. The Clean, Closed-for-Modification Validator
class Validator:
    def __init__(self, rules: list[DQRule]):
        self.rules = rules  # Validates any collection of rules dynamically

    def validate(self, records: list) -> list:
        results = []
        for rule in self.rules:
            # The validator doesn't care WHAT the rule does, 
            # it just calls the unified interface method.
            results.append(rule.evaluate(records))
        return results
    

# now show it work
# Example usage
sample_records = [
    {"email": "alice@example.com"},
    {"email": "bob[at]example.com"},
    {"email": "carol@example.com"}
]

email_rule = RegexRule("email", r"^[\w\.-]+@[\w\.-]+\.\w+$")
validator = Validator([email_rule])
display(validator.validate(sample_records))

column,failures,rule
email,1,RegexCheck


In [0]:
import traceback

class Validator:
    def __init__(self, rules: list):
        self.rules = rules

    def validate(self, records: list) -> list:
        results = []
        
        for rule in self.rules:
            try:
                rule_result = rule.evaluate(records)
                results.append(rule_result)
            except Exception as e:
                error_summary = {
                    "rule": rule.__class__.__name__,
                    "column": getattr(rule, "column", "unknown"),
                    "status": "ERROR",
                    "error_type": e.__class__.__name__,
                    "error_message": str(e),
                    "traceback": traceback.format_exc()
                }
                results.append(error_summary)
                print(f"CRITICAL: Rule {rule.__class__.__name__} failed on column {error_summary['column']}. Error: {str(e)}")
        return results

# Example rule that will raise an error
class FailingRule:
    def __init__(self, column):
        self.column = column
    def evaluate(self, records: list) -> dict:
        raise ValueError("Simulated failure for demonstration")

sample_records = [
    {"email": "alice@example.com"},
    {"email": "bob[at]example.com"},
    {"email": "carol@example.com"}
]

class RegexRule:
    def __init__(self, column: str, pattern: str):
        import re
        self.column = column
        self.pattern = re.compile(pattern)

    def evaluate(self, records: list) -> dict:
        failures = 0
        for r in records:
            v = str(r.get(self.column, ""))
            if not self.pattern.match(v):
                failures += 1
        return {"rule": "RegexCheck", "column": self.column, "failures": failures}

email_rule = RegexRule("email", r"^[\w\.-]+@[\w\.-]+\.\w+$")
failing_rule = FailingRule("email")
validator = Validator([email_rule, failing_rule])
display(validator.validate(sample_records))

CRITICAL: Rule FailingRule failed on column email. Error: Simulated failure for demonstration


column,failures,rule,error_message,error_type,status,traceback
email,1,RegexCheck,null,null,null,null
email,null,FailingRule,Simulated failure for demonstration,ValueError,ERROR,"Traceback (most recent call last): File ""/home/spark-96dcad8b-e043-46b1-aad3-51/.ipykernel/9799/command-7247235874326460-1660498050"", line 12, in validate rule_result = rule.evaluate(records) ^^^^^^^^^^^^^^^^^^^^^^ File ""/home/spark-96dcad8b-e043-46b1-aad3-51/.ipykernel/9799/command-7247235874326460-1660498050"", line 32, in evaluate raise ValueError(""Simulated failure for demonstration"") ValueError: Simulated failure for demonstration"


In [0]:
# silent py genie bugs

# Bug 1: Forgetting functools.wraps
# ❌ The Broken AI-Generated Version
# Notice that it wraps the function, but it fails to preserve the metadata of the original function.

import time

def naive_retry(attempts=3):
    def decorator(func):
        # ⚠️ BUG: @functools.wraps(func) is missing here!
        def wrapper(*args, **kwargs):
            for i in range(attempts):
                try:
                    return func(*args, **kwargs)
                except Exception:
                    if i == attempts - 1: raise
                    time.sleep(0.1)
        return wrapper
    return decorator

@naive_retry()
def fetch_api_data():
    """Fetches transactional data from external API endpoint."""
    return {"data": 123}


print(fetch_api_data.__name__) 
# Output: "wrapper"  <-- Lost the real name 'fetch_api_data'!

print(fetch_api_data.__doc__)  
# Output: None       <-- The documentation string completely vanished!

wrapper
None


In [0]:
# Bug 2: Swallowing/Masking the Exception State
# ❌ The Broken AI-Generated Version
# In this version, the AI attempts to handle the loop but mishandles the terminal exit block, completely losing the original exception context.

def broken_exception_retry(attempts=3):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for i in range(attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {i+1} failed...")
                    time.sleep(0.1)
            
            # ⚠️ BUG: The loop finished, all attempts failed. 
            # But nothing is raised or handled here! 
            # Python implicitly returns None.

        return wrapper
    return decorator

@broken_exception_retry(attempts=2)
def unstable_database_call():
    raise ConnectionRefusedError("Database is down!")

# You expect a dictionary or an explicit crash...
result = unstable_database_call()

print(result) 
# Output: None  <-- No crash! The pipeline keeps moving with dirty data.

# ...until hours later downstream:
print(result.get("status"))
# 💥 Crash: AttributeError: 'NoneType' object has no attribute 'get'

Attempt 1 failed...
Attempt 2 failed...
None


---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-7247235874326462>, line 33
     29 print(result) 
     30 # Output: None  <-- No crash! The pipeline keeps moving with dirty data.
     31 
     32 # ...until hours later downstream:
---> 33 print(result.get("status"))
     34 # 💥 Crash: AttributeError: 'NoneType' object has no attribute 'get'

AttributeError: 'NoneType' object has no attribute 'get'

In [0]:


# 🛡️ The Correct, Production-Grade Fixed Version
# This implementation fixes both bugs: it applies @functools.wraps to safely preserve metadata, and it properly retains and reraises the last seen exception if all retry thresholds are exhausted.

import functools
import time

def safe_retry(attempts=3, delay=0.1):
    def decorator(func):
        # ✅ Fix 1: Preserves original __name__, __doc__, and signature
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            
            for i in range(attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    print(f"Warning: {func.__name__} failed (Attempt {i+1}/{attempts}). Retrying...")
                    if i < attempts - 1:
                        time.sleep(delay)
            
            # ✅ Fix 2: Explicitly reraise the actual terminal error if all attempts fail
            raise last_exception
            
        return wrapper
    return decorator

@safe_retry(attempts=3, delay=0.2)
def secure_api_call():
    """Production call to fetch upstream dependencies."""
    raise TimeoutError("Gateway timeout!")



# Identity Check
print(secure_api_call.__name__) # Output: "secure_api_call" (Passed!)
print(secure_api_call.__doc__)  # Output: "Production call..." (Passed!)

# Exception Check
secure_api_call() 
# 💥 Instantly crashes with: TimeoutError: Gateway timeout! (Passed!)

secure_api_call
Production call to fetch upstream dependencies.


---------------------------------------------------------------------------
TimeoutError                              Traceback (most recent call last)
File <command-7247235874326463>, line 43
     40 print(secure_api_call.__doc__)  # Output: "Production call..." (Passed!)
     42 # Exception Check
---> 43 secure_api_call() 
     44 # 💥 Instantly crashes with: TimeoutError: Gateway timeout! (Passed!)

File <command-7247235874326463>, line 26, in safe_retry.<locals>.decorator.<locals>.wrapper(*args, **kwargs)
     23             time.sleep(delay)
     25 # ✅ Fix 2: Explicitly reraise the actual terminal error if all attempts fail
---> 26 raise last_exception

File <command-7247235874326463>, line 18, in safe_retry.<locals>.decorator.<locals>.wrapper(*args, **kwargs)
     16 for i in range(attempts):
     17     try:
---> 18         return func(*args, **kwargs)
     19     except Exception as e:
     20         last_exception = e

File <command-7247235874326463>, line 34, in secure_api_c